# [실습문제] TMDB5000 장르 다중레이블 예측하기

https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata

데이터셋은 [여기](https://github.com/user-attachments/files/20791799/tmdb_5000_movies.csv)에서 다운로드하세요 :)

위 TMDB 영화 데이터를 활용하여 다음 요구사항을 차례로 구현하고, overview를 통해 genre 멀티라벨을 예측하는 모델을 완성하세요. 

## 1. 데이터셋을 불러오고, 장르 정보를 리스트로 변환하는 코드를 작성하시오.

In [23]:
import pandas as pd
data = pd.read_csv('tmdb_5000_movies.csv')
data_df = pd.DataFrame(data)
data_df.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [24]:
import ast
# print(type(data_df['genres']))
genres = data_df['genres'].apply(
    lambda x: [d["name"] for d in ast.literal_eval(x)]
)

print(genres)

0       [Action, Adventure, Fantasy, Science Fiction]
1                        [Adventure, Fantasy, Action]
2                          [Action, Adventure, Crime]
3                    [Action, Crime, Drama, Thriller]
4                [Action, Adventure, Science Fiction]
                            ...                      
4798                        [Action, Crime, Thriller]
4799                                [Comedy, Romance]
4800               [Comedy, Drama, Romance, TV Movie]
4801                                               []
4802                                    [Documentary]
Name: genres, Length: 4803, dtype: object


## 2. 줄거리(overview) 결측치와 장르 없는 영화를 제거하는 코드를 작성하시오.

In [26]:
data_df = data_df.dropna(subset=['overview'])
data_df = data_df.dropna(subset=['genres'])

data_df.isnull().sum()        

budget                     0
genres                     0
homepage                3088
id                         0
keywords                   0
original_language          0
original_title             0
overview                   0
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    0
spoken_languages           0
status                     0
tagline                  841
title                      0
vote_average               0
vote_count                 0
dtype: int64

## 3. 장르 정보를 다중 레이블 이진화하고, 줄거리 텍스트를 TF-IDF 벡터로 변환하는 코드를 작성하시오.

In [27]:
from sklearn.preprocessing import MultiLabelBinarizer
import torch

mlb = MultiLabelBinarizer()
y = mlb.fit_transform(data_df['genres'])
y=torch.tensor(y,dtype = torch.float)
y

tensor([[1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 0., 1., 1.],
        ...,
        [1., 1., 1.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 1., 1., 1.]])

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

features = tfidf_vectorizer.fit_transform(data_df['overview'])
print(features) 

features_names = tfidf_vectorizer.get_feature_names_out()
features_names

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 197368 stored elements and shape (4800, 21262)>
  Coords	Values
  (0, 9590)	0.05634038626383085
  (0, 19006)	0.08513895439812648
  (0, 225)	0.2921220323906138
  (0, 3286)	0.1901106025470095
  (0, 13860)	0.3261456618863078
  (0, 11801)	0.24668090718542912
  (0, 10083)	0.06677258304117957
  (0, 5579)	0.26638415025671985
  (0, 19215)	0.04723852000776848
  (0, 12551)	0.25961420172221084
  (0, 13824)	0.279628173928276
  (0, 13433)	0.0798622831314405
  (0, 19918)	0.2309041805772125
  (0, 12409)	0.1700173732886173
  (0, 2856)	0.09389333593587472
  (0, 1929)	0.14627171914721604
  (0, 19294)	0.22815240966048808
  (0, 2108)	0.15333520213459115
  (0, 7502)	0.20588732915522617
  (0, 13526)	0.24255088636586541
  (0, 981)	0.04825758870048217
  (0, 14938)	0.25389029179438205
  (0, 955)	0.08040400502627923
  (0, 802)	0.199238925458924
  (0, 3599)	0.2628451007694489
  :	:
  (4799, 7341)	0.08787928740345698
  (4799, 185)	0.13061755008594614
 

array(['00', '000', '007', ..., 'été', 'única', 'über'],
      shape=(21262,), dtype=object)

## 4. 훈련/테스트 데이터로 분할 후, 다중 레이블 분류 모델을 학습하는 코드를 작성하시오.

## 5. 테스트 데이터에 대해 예측을 수행하고, 평가 결과(classification_report)를 출력하시오.

## 6. 아래의 줄거리를 입력하여 예측된 장르를 출력하시오.
`"In the future, a robot assassin is sent back in time to kill a woman."`